# EDA 08 — Connectivité & Télétravail (ARCEP + INSEE)
**Sources** :
- ARCEP Mon Réseau Mobile — couverture 4G/5G par commune × opérateur
- ARCEP Déploiements Fibre FttH — locaux éligibles par commune
- INSEE RP2020 — répartition des logements par taille (T1-T5+)

**Bronze paths** :
- `data/bronze/arcep_mobile/date=YYYY-MM-DD/part-0.parquet`
- `data/bronze/arcep_fibre/date=YYYY-MM-DD/part-0.parquet`
- `data/bronze/insee_logements/date=YYYY-MM-DD/part-0.parquet`

## Schéma Bronze — ARCEP Mobile
| `commune_code` | `arrondissement` | `operateur` | `has_4g` | `has_5g` | `pct_pop_4g` | `pct_pop_5g` | `periode` |

## Schéma Bronze — ARCEP Fibre
| `commune_code` | `arrondissement` | `nb_local_ftth` | `nb_local_total` | `pct_eligible_ftth` | `trimestre` |

## Schéma Bronze — INSEE Logements
| `commune_code` | `arrondissement` | `nb_logements_total` | `nb_t1`..`nb_t5_plus` | `pct_t2_t3` |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")

def load_latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()

df_mobile = load_latest(f"{BRONZE_ROOT}/arcep_mobile/**/*.parquet")
df_fibre  = load_latest(f"{BRONZE_ROOT}/arcep_fibre/**/*.parquet")
df_log    = load_latest(f"{BRONZE_ROOT}/insee_logements/**/*.parquet")

print(f"Mobile  : {df_mobile.shape}")
print(f"Fibre   : {df_fibre.shape}")
print(f"Logements: {df_log.shape}")


In [ ]:
# ── Génération de données synthétiques si absentes ───────────────────────────
rng = np.random.default_rng(42)
operateurs = ["orange", "sfr", "bouygues", "free"]

if df_mobile.empty:
    rows = []
    for arr in range(1, 21):
        for op in operateurs:
            pct4g = min(100, max(75, rng.normal(95, 5)))
            rows.append({
                "commune_code": f"751{arr:02d}", "arrondissement": arr,
                "operateur": op, "has_4g": True, "has_5g": arr <= 12,
                "pct_pop_4g": pct4g, "pct_pop_5g": pct4g * 0.6 if arr <= 12 else 0,
                "periode": "2024-T3", "ingested_at": pd.Timestamp("now", tz="UTC"),
            })
    df_mobile = pd.DataFrame(rows)
    print("⚠️  Mobile synthétique")

if df_fibre.empty:
    df_fibre = pd.DataFrame([{
        "commune_code": f"751{arr:02d}", "arrondissement": arr,
        "nb_local_ftth": int(rng.uniform(8000, 25000)),
        "nb_local_total": 25000,
        "pct_eligible_ftth": min(100, max(70, rng.normal(90, 8))),
        "trimestre": "2024-T3", "ingested_at": pd.Timestamp("now", tz="UTC"),
    } for arr in range(1, 21)])
    print("⚠️  Fibre synthétique")

if df_log.empty:
    df_log = pd.DataFrame([{
        "commune_code": f"751{arr:02d}", "arrondissement": arr,
        "nb_logements_total": int(rng.uniform(10000, 60000)),
        "nb_t1": int(rng.uniform(1000, 8000)), "nb_t2": int(rng.uniform(3000, 15000)),
        "nb_t3": int(rng.uniform(2000, 12000)), "nb_t4": int(rng.uniform(1000, 8000)),
        "nb_t5_plus": int(rng.uniform(500, 4000)),
        "pct_t2_t3": rng.uniform(35, 55), "annee_ref": 2020,
        "ingested_at": pd.Timestamp("now", tz="UTC"),
    } for arr in range(1, 21)])
    print("⚠️  Logements synthétique")


In [ ]:
# ── Couverture 4G par opérateur ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

op_coverage = df_mobile.groupby("operateur")["pct_pop_4g"].mean().sort_values(ascending=False)
colors = ["#FF6F00","#E53935","#00897B","#1E88E5"]
axes[0].bar(op_coverage.index, op_coverage.values, color=colors[:len(op_coverage)])
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Couverture 4G moyenne (%)")
axes[0].set_title("Couverture 4G moyenne par opérateur — Paris")
for i, (op, val) in enumerate(op_coverage.items()):
    axes[0].text(i, val + 0.5, f"{val:.1f}%", ha="center", fontsize=10)

arr_4g = df_mobile.groupby("arrondissement")["pct_pop_4g"].mean().sort_values()
colors_arr = plt.cm.RdYlGn(arr_4g.values / 100)
axes[1].barh(arr_4g.index.astype(str), arr_4g.values, color=colors_arr)
axes[1].set_xlabel("Couverture 4G moyenne (%)")
axes[1].set_title("Couverture 4G par arrondissement (moy. opérateurs)")
axes[1].set_xlim(0, 105)
plt.tight_layout()
plt.show()


In [ ]:
# ── Taux d'éligibilité Fibre ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
df_fibre_sorted = df_fibre.sort_values("pct_eligible_ftth", ascending=False)
colors = plt.cm.Blues(df_fibre_sorted["pct_eligible_ftth"] / 100)
ax.bar(df_fibre_sorted["arrondissement"].astype(str), df_fibre_sorted["pct_eligible_ftth"], color=colors)
ax.set_xlabel("Arrondissement")
ax.set_ylabel("% locaux éligibles FttH")
ax.set_title("Taux d'éligibilité à la fibre optique par arrondissement")
ax.set_ylim(0, 105)
for i, (_, row) in enumerate(df_fibre_sorted.iterrows()):
    ax.text(i, row["pct_eligible_ftth"] + 0.5, f"{row['pct_eligible_ftth']:.0f}%",
            ha="center", fontsize=8)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()
print(f"Taux moyen d'éligibilité fibre : {df_fibre['pct_eligible_ftth'].mean():.1f}%")


In [ ]:
# ── Répartition des logements T2/T3 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_log_sorted = df_log.sort_values("pct_t2_t3", ascending=False)
colors = plt.cm.Greens(df_log_sorted["pct_t2_t3"] / df_log_sorted["pct_t2_t3"].max())
axes[0].bar(df_log_sorted["arrondissement"].astype(str), df_log_sorted["pct_t2_t3"], color=colors)
axes[0].set_xlabel("Arrondissement")
axes[0].set_ylabel("% logements T2/T3")
axes[0].set_title("Proportion logements T2/T3 par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Stacked bar par type
types = ["nb_t1","nb_t2","nb_t3","nb_t4","nb_t5_plus"]
labels = ["T1","T2","T3","T4","T5+"]
colors_t = ["#EF9A9A","#90CAF9","#A5D6A7","#FFE082","#CE93D8"]
bottom = np.zeros(20)
for col, label, color in zip(types, labels, colors_t):
    axes[1].bar(df_log.sort_values("arrondissement")["arrondissement"].astype(str),
                df_log.sort_values("arrondissement")[col], bottom=bottom, label=label, color=color)
    bottom += df_log.sort_values("arrondissement")[col].values
axes[1].set_xlabel("Arrondissement")
axes[1].set_ylabel("Logements")
axes[1].set_title("Répartition par taille de logement")
axes[1].legend(loc="upper right")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()


## Conclusions EDA
- La fibre est quasi-universelle à Paris (>90% dans la plupart des arrondissements), la différenciation se fait sur la 5G.
- Les 4 opérateurs majeurs couvrent uniformément Paris en 4G.
- Les arrondissements centraux ont plus de T1/T2 (logements étudiants/jeunes), l'Ouest plus de grands appartements.
- Le score `connectivity` = 40% fibre + 30% 4G + 30% proportion T2/T3 → indicateur "capacité télétravail".
